In [35]:
from datasets import load_dataset

dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
)

print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response'],
        num_rows: 26872
    })
})
{'flags': 'B', 'instruction': 'question about cancelling order {{Order Number}}', 'category': 'ORDER', 'intent': 'cancel_order', 'response': "I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you."}


In [36]:
print("Dataset splits:", dataset.keys())

train_data = dataset["train"]

print("\nNumber of records:", len(train_data))
print("\nColumn names:", train_data.column_names)
print("\nFirst record:")
print(train_data[0])

print("\nFeatures:")
print(train_data.features)

Dataset splits: dict_keys(['train'])

Number of records: 26872

Column names: ['flags', 'instruction', 'category', 'intent', 'response']

First record:
{'flags': 'B', 'instruction': 'question about cancelling order {{Order Number}}', 'category': 'ORDER', 'intent': 'cancel_order', 'response': "I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you."}

Features:
{'flags': Value('string'), 'instruction': Value('string'), 'category': Value('string'), 'intent': Value('string'), 'response': Value('string')}


In [37]:
from collections import Counter

print("Unique categories:")
print(Counter(train_data["category"]))

print("\nNumber of unique intents:")
print(len(set(train_data["intent"])))

print("\nExample intents:")
print(sorted(set(train_data["intent"]))[:30])

Unique categories:
Counter({'ACCOUNT': 5986, 'ORDER': 3988, 'REFUND': 2992, 'INVOICE': 1999, 'CONTACT': 1999, 'PAYMENT': 1998, 'FEEDBACK': 1997, 'DELIVERY': 1994, 'SHIPPING': 1970, 'SUBSCRIPTION': 999, 'CANCEL': 950})

Number of unique intents:
27

Example intents:
['cancel_order', 'change_order', 'change_shipping_address', 'check_cancellation_fee', 'check_invoice', 'check_payment_methods', 'check_refund_policy', 'complaint', 'contact_customer_service', 'contact_human_agent', 'create_account', 'delete_account', 'delivery_options', 'delivery_period', 'edit_account', 'get_invoice', 'get_refund', 'newsletter_subscription', 'payment_issue', 'place_order', 'recover_password', 'registration_problems', 'review', 'set_up_shipping_address', 'switch_account', 'track_order', 'track_refund']


In [38]:
from collections import Counter

print("Flags:", Counter(train_data["flags"]))
print("\nSample by category:")

for category in ["ACCOUNT", "ORDER", "PAYMENT", "REFUND", "DELIVERY"]:
    row = next(
        r for r in train_data
        if r["category"] == category
    )
    print(f"\n{category}")
    print("Instruction:", row["instruction"])
    print("Intent:", row["intent"])
    print("Response:", row["response"][:200])

Flags: Counter({'BL': 5212, 'BLQ': 2467, 'BIL': 2138, 'BLM': 1297, 'BILQ': 1057, 'BLQZ': 970, 'BLZ': 902, 'BKL': 862, 'BLMQ': 600, 'BEL': 533, 'BILM': 521, 'BCL': 484, 'BCIL': 427, 'BILQZ': 412, 'BILP': 363, 'B': 353, 'BK': 352, 'BQ': 340, 'BKLZ': 322, 'BLW': 309, 'BILZ': 279, 'BKLM': 244, 'BELQ': 242, 'BILMQ': 230, 'BLMQZ': 198, 'BLMZ': 192, 'BI': 160, 'BILW': 147, 'BIQ': 146, 'BQZ': 143, 'BCILP': 142, 'BKLQ': 132, 'BCILQ': 125, 'BZ': 120, 'BELM': 120, 'BCLM': 109, 'BLMW': 105, 'BELQZ': 104, 'BKM': 100, 'BLP': 91, 'BELZ': 91, 'BCLQZ': 88, 'BKLMZ': 87, 'BILMQZ': 82, 'BILMP': 81, 'BLN': 80, 'BCILQZ': 80, 'BCLQ': 77, 'BCEL': 75, 'BILMZ': 71, 'BCILM': 70, 'BIS': 70, 'BIQS': 70, 'BKZ': 64, 'BELMQ': 60, 'BISZ': 59, 'BIQSZ': 58, 'BLQW': 56, 'BCLW': 54, 'BMQ': 51, 'BEQ': 50, 'BLQWZ': 48, 'BILPQ': 46, 'BELN': 44, 'BIQZ': 43, 'BCILW': 41, 'BIZ': 39, 'BKLQZ': 39, 'BCILPQZ': 38, 'BILPQZ': 36, 'BLNQ': 36, 'BCEIL': 36, 'BM': 34, 'BILMW': 34, 'BILPW': 34, 'BCLZ': 34, 'BELP': 33, 'BE': 33, 'BELW': 30

In [39]:

import pandas as pd
import re

df = dataset["train"].to_pandas()

# Normalize text fields
for col in ["instruction", "response", "category", "intent"]:
    df[col] = df[col].fillna("").astype(str).str.strip()

# Remove empty records and exact duplicate examples
df = df[
    (df["instruction"] != "") &
    (df["response"] != "")
].drop_duplicates(subset=["instruction", "response"]).copy()

# Map source categories to Sortly AI categories
def map_category(row):
    category = row["category"].upper()
    intent = row["intent"].lower()

    if category in {"PAYMENT", "INVOICE"}:
        return "Payment"

    if category == "REFUND":
        return "Refund"

    if category in {"DELIVERY", "SHIPPING"}:
        return "Delivery"

    if category == "ACCOUNT":
        if any(x in intent for x in [
            "recover_password", "login", "sign_in", "password"
        ]):
            return "Login"
        return "Account Management"

    if category in {"ORDER", "CANCEL"}:
        if "refund" in intent:
            return "Refund"
        if any(x in intent for x in [
            "track_order", "shipping", "delivery"
        ]):
            return "Delivery"
        return "Account Management"

    if category in {"CONTACT", "FEEDBACK"}:
        return "Technical Issue"

    return None

df["target_category"] = df.apply(map_category, axis=1)
df = df.dropna(subset=["target_category"]).copy()

print("Prepared records:", len(df))
print("\nTarget categories:")
print(df["target_category"].value_counts())


Prepared records: 25873

Target categories:
target_category
Account Management    8934
Delivery              4959
Payment               3997
Technical Issue       3996
Refund                2992
Login                  995
Name: count, dtype: int64


In [40]:

# Create weak labels from available text and intent.
# These rules are a starting point and must be validated.

def assign_priority(row):
    text = (row["instruction"] + " " + row["intent"]).lower()

    high_terms = [
        "urgent", "fraud", "unauthorized", "charged twice",
        "payment_issue", "cannot access"
    ]
    low_terms = ["newsletter", "review", "feedback"]

    if any(term in text for term in high_terms):
        return "High"
    if any(term in text for term in low_terms):
        return "Low"
    return "Medium"


def assign_sentiment(row):
    text = (row["instruction"] + " " + row["response"]).lower()

    negative_terms = [
        "complaint", "problem", "issue", "failed",
        "cannot", "can't", "error", "wrong"
    ]
    positive_terms = [
        "thank", "great", "excellent", "happy", "love"
    ]

    if any(term in text for term in negative_terms):
        return "Negative"
    if any(term in text for term in positive_terms):
        return "Positive"
    return "Neutral"


df["priority"] = df.apply(assign_priority, axis=1)
df["sentiment"] = df.apply(assign_sentiment, axis=1)

print("Priority distribution:")
print(df["priority"].value_counts())

print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

print("\nSample records:")
print(df[[
    "instruction", "target_category", "priority", "sentiment"
]].head(10).to_string(index=False))


Priority distribution:
priority
Medium    23877
High        999
Low         997
Name: count, dtype: int64

Sentiment distribution:
sentiment
Neutral     11901
Positive     7309
Negative     6663
Name: count, dtype: int64

Sample records:
                                                 instruction    target_category priority sentiment
            question about cancelling order {{Order Number}} Account Management   Medium   Neutral
  i have a question about cancelling oorder {{Order Number}} Account Management   Medium   Neutral
             i need help cancelling puchase {{Order Number}} Account Management   Medium   Neutral
                  I need to cancel purchase {{Order Number}} Account Management   Medium   Neutral
I cannot afford this order, cancel purchase {{Order Number}} Account Management   Medium  Negative
              can you help me cancel order {{Order Number}}? Account Management   Medium   Neutral
    I can no longer afford order {{Order Number}}, cancel it Account 

In [41]:

def map_category(row):
    category = row["category"].upper()
    intent = row["intent"].lower()

    if category == "REFUND" or "refund" in intent:
        return "Refund"

    if category in {"PAYMENT", "INVOICE"} or "payment" in intent:
        return "Payment"

    if category in {"DELIVERY", "SHIPPING"} or any(
        x in intent for x in ["track_order", "shipping", "delivery"]
    ):
        return "Delivery"

    if any(x in intent for x in [
        "recover_password", "login", "sign_in", "password"
    ]):
        return "Login"

    if category in {"ORDER", "CANCEL"} or any(
        x in intent for x in ["cancel_order", "change_order", "place_order"]
    ):
        return "Account Management"  # No Order category in our six-class schema

    if category == "ACCOUNT":
        return "Account Management"

    if category in {"CONTACT", "FEEDBACK"}:
        return "Technical Issue"

    return None

df["target_category"] = df.apply(map_category, axis=1)
df = df.dropna(subset=["target_category"]).copy()

print("Records:", len(df))
print(df["target_category"].value_counts())



Records: 25873
target_category
Account Management    8934
Delivery              4959
Payment               3997
Technical Issue       3996
Refund                2992
Login                  995
Name: count, dtype: int64


In [42]:

from sklearn.model_selection import train_test_split

# Create a stratified split using the six target categories
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["target_category"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["target_category"]
)

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nTraining category distribution:")
print(train_df["target_category"].value_counts())


Training: 20698
Validation: 2587
Test: 2588

Training category distribution:
target_category
Account Management    7147
Delivery              3967
Payment               3197
Technical Issue       3197
Refund                2394
Login                  796
Name: count, dtype: int64


In [43]:
from pathlib import Path

output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

train_df.to_csv(output_dir / "train.csv", index=False)
val_df.to_csv(output_dir / "validation.csv", index=False)
test_df.to_csv(output_dir / "test.csv", index=False)

print("Saved files:")
for file in output_dir.glob("*.csv"):
    print(file, file.stat().st_size, "bytes")

Saved files:
data/processed/train.csv 15428968 bytes
data/processed/validation.csv 1942165 bytes
data/processed/test.csv 1930538 bytes


In [44]:

from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from pathlib import Path
import pandas as pd

# Load the training split we saved
train_df = pd.read_csv("data/processed/train.csv")

# Initialize a BPE tokenizer from scratch
tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=2000,
    min_frequency=2,
    special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet()
)

# Train only on the training split
tokenizer.train_from_iterator(
    train_df["instruction"].astype(str).tolist(),
    trainer=trainer
)

# Save tokenizer files
output_dir = Path("tokenizer")
output_dir.mkdir(parents=True, exist_ok=True)
tokenizer.save(str(output_dir / "tokenizer.json"))

print("Tokenizer vocabulary size:", tokenizer.get_vocab_size())

# Test tokenization
sample = "I cannot access my account after payment."
encoded = tokenizer.encode(sample)

print("Original text:", sample)
print("Tokens:", encoded.tokens)
print("Token IDs:", encoded.ids)
print("Decoded:", tokenizer.decode(encoded.ids))


Tokenizer vocabulary size: 2000
Original text: I cannot access my account after payment.
Tokens: ['I', 'Ġcannot', 'Ġaccess', 'Ġmy', 'Ġaccount', 'Ġaf', 't', 'er', 'Ġpayment', '.']
Token IDs: [44, 707, 909, 305, 321, 1092, 87, 262, 379, 17]
Decoded: I cannot access my account after payment.


In [45]:

from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from pathlib import Path

tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel()
tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=2000,
    min_frequency=2,
    special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet()
)

def text_corpus():
    for row in train_df.itertuples():
        yield str(row.instruction)
        yield str(row.response)

tokenizer.train_from_iterator(text_corpus(), trainer=trainer)

Path("tokenizer").mkdir(exist_ok=True)
tokenizer.save("tokenizer/tokenizer.json")

print("Vocabulary size:", tokenizer.get_vocab_size())
print(tokenizer.encode(
    "I need help with my payment."
).tokens)


Vocabulary size: 2000
['ĠI', 'Ġneed', 'Ġhelp', 'Ġwith', 'Ġmy', 'Ġpayment', '.']


In [46]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("tokenizer/tokenizer.json")

for token in ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]:
    print(token, "->", tokenizer.token_to_id(token))

[PAD] -> 0
[UNK] -> 1
[BOS] -> 2
[EOS] -> 3


In [47]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# 1. Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# 2. Model configuration
VOCAB_SIZE = tokenizer.get_vocab_size()  # 2,000 tokens
MAX_SEQ_LEN = 128
D_MODEL = 128
N_HEADS = 4
N_LAYERS = 4
D_FF = 512
DROPOUT = 0.1

# 3. Causal self-attention
class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        assert D_MODEL % N_HEADS == 0
        self.attention = nn.MultiheadAttention(
            D_MODEL, N_HEADS,
            dropout=DROPOUT,
            batch_first=True
        )
        self.register_buffer(
            "causal_mask",
            torch.triu(
                torch.ones(MAX_SEQ_LEN, MAX_SEQ_LEN, dtype=torch.bool),
                diagonal=1
            )
        )

    def forward(self, x):
        length = x.size(1)
        mask = self.causal_mask[:length, :length]
        output, _ = self.attention(
            x, x, x,
            attn_mask=mask,
            need_weights=False
        )
        return output

# 4. Transformer block
class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.norm1 = nn.LayerNorm(D_MODEL)
        self.attn = CausalSelfAttention()
        self.norm2 = nn.LayerNorm(D_MODEL)
        self.ff = nn.Sequential(
            nn.Linear(D_MODEL, D_FF),
            nn.GELU(),
            nn.Linear(D_FF, D_MODEL),
            nn.Dropout(DROPOUT)
        )
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.norm1(x)))
        x = x + self.dropout(self.ff(self.norm2(x)))
        return x

# 5. Sortly AI language model
class SortlyTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(VOCAB_SIZE, D_MODEL)
        self.position_embedding = nn.Embedding(MAX_SEQ_LEN, D_MODEL)
        self.blocks = nn.Sequential(
            *[TransformerBlock() for _ in range(N_LAYERS)]
        )
        self.final_norm = nn.LayerNorm(D_MODEL)
        self.lm_head = nn.Linear(D_MODEL, VOCAB_SIZE, bias=False)

        # Share token embeddings with output projection
        self.lm_head.weight = self.token_embedding.weight

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape
        positions = torch.arange(seq_len, device=input_ids.device)
        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )
        x = self.blocks(x)
        x = self.final_norm(x)
        return self.lm_head(x)

# 6. Initialize model from random weights
model = SortlyTransformer().to(device)

parameter_count = sum(p.numel() for p in model.parameters())
print(f"Vocabulary size: {VOCAB_SIZE}")
print(f"Parameters: {parameter_count:,}")
print(f"Model device: {next(model.parameters()).device}")

# 7. Test a forward pass
sample_ids = tokenizer.encode(
    "I need help with my payment."
).ids
sample_ids = sample_ids[:MAX_SEQ_LEN]

input_tensor = torch.tensor(
    [sample_ids], dtype=torch.long, device=device
)

with torch.no_grad():
    logits = model(input_tensor)

print("Input shape:", tuple(input_tensor.shape))
print("Output shape:", tuple(logits.shape))
print("Transformer forward pass successful!")


Device: cuda
Vocabulary size: 2000
Parameters: 1,065,728
Model device: cuda:0
Input shape: (1, 7)
Output shape: (1, 7, 2000)
Transformer forward pass successful!


In [48]:

import torch
from torch.utils.data import Dataset, DataLoader

PAD_ID = tokenizer.token_to_id("[PAD]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")

class SupportDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.examples = []
        self.max_length = max_length

        for row in dataframe.itertuples():
            # Train on the customer question and its response
            text = f"Customer: {row.instruction}\nAgent: {row.response}"
            ids = tokenizer.encode(text).ids

            # Reserve space for beginning and end markers
            ids = [BOS_ID] + ids[:max_length - 2] + [EOS_ID]
            self.examples.append(ids)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        ids = self.examples[index]
        input_ids = ids[:-1]
        targets = ids[1:]

        # Pad both sequences to a fixed length
        input_ids = input_ids + [PAD_ID] * (self.max_length - len(input_ids))
        targets = targets + [-100] * (self.max_length - len(targets))

        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(targets, dtype=torch.long)
        )

train_dataset = SupportDataset(train_df, tokenizer)
val_dataset = SupportDataset(val_df, tokenizer)

train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True
)
val_loader = DataLoader(
    val_dataset, batch_size=32, shuffle=False
)

# Verify a batch
x, y = next(iter(train_loader))

print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("Batch input shape:", tuple(x.shape))
print("Batch target shape:", tuple(y.shape))
print("Padding token ID:", PAD_ID)
print("Ignored target positions:", (y == -100).sum().item())


Training examples: 20698
Validation examples: 2587
Batch input shape: (32, 128)
Batch target shape: (32, 128)
Padding token ID: 0
Ignored target positions: 317


In [49]:

import torch
import torch.nn as nn
from pathlib import Path

# Training configuration
EPOCHS = 5
LEARNING_RATE = 3e-4

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01
)

criterion = nn.CrossEntropyLoss(ignore_index=-100)
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

Path("checkpoints").mkdir(exist_ok=True)
best_val_loss = float("inf")

def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    total_batches = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device == "cuda")
            ):
                logits = model(x)
                loss = criterion(
                    logits.reshape(-1, VOCAB_SIZE),
                    y.reshape(-1)
                )

            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), max_norm=1.0
                )
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item()
        total_batches += 1

    return total_loss / max(total_batches, 1)

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_loader, training=True)
    val_loss = run_epoch(val_loader, training=False)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Validation Loss: {val_loss:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "config": {
                    "vocab_size": VOCAB_SIZE,
                    "max_seq_len": MAX_SEQ_LEN,
                    "d_model": D_MODEL,
                    "n_heads": N_HEADS,
                    "n_layers": N_LAYERS,
                    "d_ff": D_FF
                },
                "epoch": epoch,
                "val_loss": val_loss
            },
            "checkpoints/sortly_transformer_best.pt"
        )
        print("  Best checkpoint saved.")

print("Training complete. Best validation loss:", best_val_loss)


Epoch 1/5 | Train Loss: 9.5144 | Validation Loss: 4.3213
  Best checkpoint saved.
Epoch 2/5 | Train Loss: 4.4098 | Validation Loss: 3.6440
  Best checkpoint saved.
Epoch 3/5 | Train Loss: 3.7856 | Validation Loss: 3.2246
  Best checkpoint saved.
Epoch 4/5 | Train Loss: 3.3619 | Validation Loss: 2.8616
  Best checkpoint saved.
Epoch 5/5 | Train Loss: 3.0259 | Validation Loss: 2.5801
  Best checkpoint saved.
Training complete. Best validation loss: 2.5800596755227927


In [50]:

import torch
import torch.nn.functional as F

# Load the best checkpoint
checkpoint = torch.load(
    "checkpoints/sortly_transformer_best.pt",
    map_location=device,
    weights_only=True
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

@torch.no_grad()
def generate_text(prompt, max_new_tokens=60, temperature=0.8):
    ids = [BOS_ID] + tokenizer.encode(prompt).ids
    input_ids = torch.tensor([ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        # Keep the most recent context within model limits
        context = input_ids[:, -MAX_SEQ_LEN:]

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device == "cuda")
        ):
            logits = model(context)

        next_logits = logits[0, -1, :].float() / temperature

        # Avoid generating padding or a new beginning marker
        next_logits[PAD_ID] = -float("inf")
        next_logits[BOS_ID] = -float("inf")

        probabilities = F.softmax(next_logits, dim=-1)
        next_id = torch.multinomial(probabilities, num_samples=1).item()

        if next_id == EOS_ID:
            break

        input_ids = torch.cat(
            [input_ids, torch.tensor([[next_id]], device=device)],
            dim=1
        )

    generated_ids = input_ids[0].tolist()[1:]
    return tokenizer.decode(generated_ids)

print(generate_text("Customer: I need help with my payment.\nAgent:"))


 Customer: I need help with my payment.
Agent: Thank you for reaching out to help! I understand the payment methods we offer for any payment methods. To better assist you with the payment methods, could you please provide us with the payment experience.


In [51]:

UNK_ID = tokenizer.token_to_id("[UNK]")

@torch.no_grad()
def generate_topk(prompt, max_new_tokens=80, temperature=0.7, top_k=20):
    model.eval()

    ids = [BOS_ID] + tokenizer.encode(prompt).ids
    input_ids = torch.tensor([ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        context = input_ids[:, -MAX_SEQ_LEN:]

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device == "cuda")
        ):
            logits = model(context)

        scores = logits[0, -1, :].float() / temperature
        scores[PAD_ID] = -float("inf")
        scores[BOS_ID] = -float("inf")
        scores[UNK_ID] = -float("inf")

        values, indices = torch.topk(
            scores, min(top_k, scores.size(-1))
        )
        probs = F.softmax(values, dim=-1)
        next_id = indices[torch.multinomial(probs, 1)].item()

        if next_id == EOS_ID:
            break

        input_ids = torch.cat([
            input_ids,
            torch.tensor([[next_id]], device=device)
        ], dim=1)

    return tokenizer.decode(input_ids[0].tolist()[1:])

print(generate_topk(
    "Customer: I need help with my payment.\nAgent:"
))


 Customer: I need help with my payment.
Agent: I'm delighted to hear that you're encountering encountering the payment in ut and we appreciate your payment methods we have the payment methods. To assist you in helping us, I'm here to help you in the assistance, could you please provide more details about the payment matter.


In [52]:

print("Train rows:", len(train_df))
print("Tokenizer vocabulary:", tokenizer.get_vocab_size())
print("\nExample customer message:")
print(train_df.iloc[0]["instruction"])
print("\nExample agent response:")
print(train_df.iloc[0]["response"])


Train rows: 20698
Tokenizer vocabulary: 2000

Example customer message:
there is an issue submitting the new delivery address

Example agent response:
I can relate to the understanding that you're encountering difficulties in submitting your new delivery address, and I'm here to assist you. To address this issue, I would recommend the following steps:

1. Double-check all the fields and ensure that you have provided the correct information for your new delivery address.
2. Make sure that there are no errors or typos in the address details, such as misspelled street names, incorrect zip codes, or missing apartment numbers.
3. Clear your browser cache and cookies and try submitting the address again. Sometimes, technical issues can interfere with the submission process.
4. If the issue persists, try using a different web browser or device to submit the new delivery address. This can help identify if the problem is specific to your current browser or device.
5. If none of the above steps 

In [53]:

from torch.utils.data import Dataset, DataLoader

class ResponseOnlyDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.examples = []
        self.max_length = max_length

        for row in dataframe.itertuples():
            prompt = f"Customer: {row.instruction}\nAgent:"
            answer = " " + str(row.response)

            prompt_ids = tokenizer.encode(prompt).ids
            answer_ids = tokenizer.encode(answer).ids

            # Reserve room for BOS, prompt, response and EOS
            prompt_ids = prompt_ids[:max_length - 3]
            remaining = max_length - 2 - len(prompt_ids)
            answer_ids = answer_ids[:max(0, remaining - 1)]

            ids = [BOS_ID] + prompt_ids + answer_ids + [EOS_ID]
            prompt_length = 1 + len(prompt_ids)

            # Ignore prompt tokens when calculating the loss
            targets = ids[1:] + [-100]
            targets = targets[:len(ids)]

            # Align targets with next-token prediction
            targets = ids[1:] + [-100]
            targets = [
                target if i >= prompt_length - 1 else -100
                for i, target in enumerate(targets)
            ]

            padding = max_length - len(ids)
            ids += [PAD_ID] * padding
            targets += [-100] * padding

            self.examples.append((ids, targets))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        ids, targets = self.examples[index]
        return (
            torch.tensor(ids[:-1], dtype=torch.long),
            torch.tensor(targets[:-1], dtype=torch.long)
        )

train_dataset = ResponseOnlyDataset(train_df, tokenizer)
val_dataset = ResponseOnlyDataset(val_df, tokenizer)

train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True
)
val_loader = DataLoader(
    val_dataset, batch_size=32, shuffle=False
)

x, y = next(iter(train_loader))
print("Input shape:", tuple(x.shape))
print("Target shape:", tuple(y.shape))
print("Ignored target positions:", (y == -100).sum().item())
print("Response-only dataset ready.")


Input shape: (32, 127)
Target shape: (32, 127)
Ignored target positions: 914
Response-only dataset ready.


In [54]:

# Continue training from the best existing checkpoint
checkpoint = torch.load(
    "checkpoints/sortly_transformer_best.pt",
    map_location=device,
    weights_only=True
)
model.load_state_dict(checkpoint["model_state_dict"])

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=0.01
)

criterion = nn.CrossEntropyLoss(ignore_index=-100)
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

best_response_val_loss = float("inf")

for epoch in range(1, 6):
    train_loss = run_epoch(train_loader, training=True)
    val_loss = run_epoch(val_loader, training=False)

    print(
        f"Response Fine-tuning {epoch}/5 | "
        f"Train Loss: {train_loss:.4f} | "
        f"Validation Loss: {val_loss:.4f}"
    )

    if val_loss < best_response_val_loss:
        best_response_val_loss = val_loss
        torch.save(
            model.state_dict(),
            "checkpoints/sortly_response_best.pt"
        )
        print("  Best response checkpoint saved.")

print("Best response validation loss:", best_response_val_loss)


Response Fine-tuning 1/5 | Train Loss: 2.9728 | Validation Loss: 2.6120
  Best response checkpoint saved.
Response Fine-tuning 2/5 | Train Loss: 2.8588 | Validation Loss: 2.5191
  Best response checkpoint saved.
Response Fine-tuning 3/5 | Train Loss: 2.7643 | Validation Loss: 2.4329
  Best response checkpoint saved.
Response Fine-tuning 4/5 | Train Loss: 2.6801 | Validation Loss: 2.3578
  Best response checkpoint saved.
Response Fine-tuning 5/5 | Train Loss: 2.5996 | Validation Loss: 2.2847
  Best response checkpoint saved.
Best response validation loss: 2.284709690529623


In [55]:

# Load the best response-focused checkpoint
model.load_state_dict(
    torch.load(
        "checkpoints/sortly_response_best.pt",
        map_location=device,
        weights_only=True
    )
)
model.eval()

prompt = "Customer: there is an issue submitting the new delivery address\nAgent:"

print(generate_topk(
    prompt,
    max_new_tokens=100,
    temperature=0.7,
    top_k=20
))


 Customer: there is an issue submitting the new delivery address
Agent: I appreciate your frustration you need to know the bill whiley. To ensure the information, I recommend visiting our website and navigate to the "Contact Us" section. If you need further questions or need further assistance, please feel free to let me know if there, feel free to let me know.


In [56]:

prompt = (
    "Customer: there is an issue submitting the new delivery address\n"
    "Agent:"
)

for temperature in [0.5, 0.7]:
    print(f"\n--- Temperature: {temperature} ---")
    print(generate_topk(
        prompt,
        max_new_tokens=100,
        temperature=temperature,
        top_k=10
    ))



--- Temperature: 0.5 ---
 Customer: there is an issue submitting the new delivery address
Agent: I'm on it! I'm here to assist you in setting up your shipping address. To assist you with the information, could you please provide me with the most accurate information such as the details about the specific information you would like to the date.

--- Temperature: 0.7 ---
 Customer: there is an issue submitting the new delivery address
Agent: I'm here to help! I'm here to assist you with that. Here's a new delivery address, and I'm here to assist you in resolving this process. To ensure a new delivery address, please login code, navigate to your account when you're logged in your account.


In [57]:

lengths = [
    len(tokenizer.encode(str(response)).ids)
    for response in train_df["response"]
]

import numpy as np

print("Training responses:", len(lengths))
print("Median response tokens:", int(np.median(lengths)))
print("95th percentile:", int(np.percentile(lengths, 95)))
print("Responses over 100 tokens:", sum(n > 100 for n in lengths))
print("Responses over 126 tokens:", sum(n > 126 for n in lengths))
print(
    "Percentage over 126 tokens:",
    round(100 * sum(n > 126 for n in lengths) / len(lengths), 2),
    "%"
)


Training responses: 20698
Median response tokens: 114
95th percentile: 298
Responses over 100 tokens: 12920
Responses over 126 tokens: 8558
Percentage over 126 tokens: 41.35 %


In [58]:

import torch

if torch.cuda.is_available():
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Free memory: {free_bytes / 1024**3:.2f} GB")
    print(f"Total memory: {total_bytes / 1024**3:.2f} GB")
else:
    print("GPU unavailable")


GPU: Tesla T4
Free memory: 14.14 GB
Total memory: 14.56 GB


In [59]:

# Update context length
MAX_SEQ_LEN = 384

# Rebuild the model with the new positional embeddings and mask
model = SortlyTransformer().to(device)

print("Context length:", MAX_SEQ_LEN)
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Device:", next(model.parameters()).device)


Context length: 384
Parameters: 1,098,496
Device: cuda:0


In [60]:

MAX_SEQ_LEN = 128

def prepare_response_examples(dataframe, tokenizer, max_length=128):
    examples = []
    skipped = 0

    for row in dataframe.itertuples():
        prompt = f"Customer: {row.instruction}\nAgent:"
        prompt_ids = tokenizer.encode(prompt).ids
        answer_ids = tokenizer.encode(" " + str(row.response)).ids

        # Reserve BOS and EOS tokens
        available = max_length - len(prompt_ids) - 2

        # Skip examples where the prompt leaves too little room
        if available < 20:
            skipped += 1
            continue

        # Keep only responses that fit completely
        if len(answer_ids) > available:
            skipped += 1
            continue

        examples.append((prompt, str(row.response)))

    print("Kept examples:", len(examples))
    print("Skipped examples:", skipped)
    return examples

clean_train_examples = prepare_response_examples(
    train_df, tokenizer, MAX_SEQ_LEN
)
clean_val_examples = prepare_response_examples(
    val_df, tokenizer, MAX_SEQ_LEN
)


Kept examples: 9192
Skipped examples: 11506
Kept examples: 1155
Skipped examples: 1432


In [61]:

from torch.utils.data import Dataset, DataLoader
import torch

class FilteredResponseDataset(Dataset):
    def __init__(self, examples, tokenizer, max_length=128):
        self.items = []
        self.max_length = max_length

        for prompt, response in examples:
            prompt_ids = tokenizer.encode(prompt).ids
            answer_ids = tokenizer.encode(" " + response).ids

            ids = [BOS_ID] + prompt_ids + answer_ids + [EOS_ID]
            prompt_length = 1 + len(prompt_ids)

            # Targets correspond to the next token
            targets = ids[1:] + [-100]
            targets = [
                target if i >= prompt_length - 1 else -100
                for i, target in enumerate(targets)
            ]

            # Pad to a fixed length
            ids = ids[:max_length]
            targets = targets[:max_length - 1]

            input_ids = ids[:-1]
            padding = (max_length - 1) - len(input_ids)
            input_ids += [PAD_ID] * padding
            targets += [-100] * ((max_length - 1) - len(targets))

            self.items.append((input_ids, targets))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        x, y = self.items[index]
        return torch.tensor(x), torch.tensor(y)

train_dataset = FilteredResponseDataset(
    clean_train_examples, tokenizer
)
val_dataset = FilteredResponseDataset(
    clean_val_examples, tokenizer
)

train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True
)
val_loader = DataLoader(
    val_dataset, batch_size=32, shuffle=False
)

x, y = next(iter(train_loader))
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Input shape:", tuple(x.shape))
print("Target shape:", tuple(y.shape))
print("Ignored targets:", (y == -100).sum().item())


Train batches: 288
Validation batches: 37
Input shape: (32, 127)
Target shape: (32, 127)
Ignored targets: 1471


In [62]:
# Restore the original 128-token architecture configuration to match the checkpoint
MAX_SEQ_LEN = 128
model = SortlyTransformer().to(device)

# Restore the best previously trained model
checkpoint = torch.load(
    "checkpoints/sortly_response_best.pt",
    map_location=device,
    weights_only=True
)
model.load_state_dict(checkpoint)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-5,
    weight_decay=0.01
)
criterion = nn.CrossEntropyLoss(ignore_index=-100)
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

best_filtered_val_loss = float("inf")

for epoch in range(1, 4):
    train_loss = run_epoch(train_loader, training=True)
    val_loss = run_epoch(val_loader, training=False)

    print(
        f"Filtered training {epoch}/3 | "
        f"Train Loss: {train_loss:.4f} | "
        f"Validation Loss: {val_loss:.4f}"
    )

    if val_loss < best_filtered_val_loss:
        best_filtered_val_loss = val_loss
        torch.save(
            model.state_dict(),
            "checkpoints/sortly_filtered_best.pt"
        )
        print("  Best filtered checkpoint saved.")

print("Best filtered validation loss:", best_filtered_val_loss)

Filtered training 1/3 | Train Loss: 2.3978 | Validation Loss: 2.1081
  Best filtered checkpoint saved.
Filtered training 2/3 | Train Loss: 2.3443 | Validation Loss: 2.0723
  Best filtered checkpoint saved.
Filtered training 3/3 | Train Loss: 2.3094 | Validation Loss: 2.0448
  Best filtered checkpoint saved.
Best filtered validation loss: 2.0447733627783284


In [63]:

# Restore the original 128-token architecture
MAX_SEQ_LEN = 128
model = SortlyTransformer().to(device)

# Load the best response-trained checkpoint
checkpoint = torch.load(
    "checkpoints/sortly_response_best.pt",
    map_location=device,
    weights_only=True
)
model.load_state_dict(checkpoint)
model.eval()

print("Model restored successfully")
print("Context length:", MAX_SEQ_LEN)
print("Device:", next(model.parameters()).device)


Model restored successfully
Context length: 128
Device: cuda:0


In [64]:

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-5,
    weight_decay=0.01
)
criterion = nn.CrossEntropyLoss(ignore_index=-100)
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

best_filtered_val_loss = float("inf")

for epoch in range(1, 4):
    train_loss = run_epoch(train_loader, training=True)
    val_loss = run_epoch(val_loader, training=False)

    print(
        f"Epoch {epoch}/3 | "
        f"Train Loss: {train_loss:.4f} | "
        f"Validation Loss: {val_loss:.4f}"
    )

    if val_loss < best_filtered_val_loss:
        best_filtered_val_loss = val_loss
        torch.save(
            model.state_dict(),
            "checkpoints/sortly_filtered_best.pt"
        )
        print("Best checkpoint saved.")

print("Best validation loss:", best_filtered_val_loss)


Epoch 1/3 | Train Loss: 2.3979 | Validation Loss: 2.1082
Best checkpoint saved.
Epoch 2/3 | Train Loss: 2.3454 | Validation Loss: 2.0746
Best checkpoint saved.
Epoch 3/3 | Train Loss: 2.3092 | Validation Loss: 2.0443
Best checkpoint saved.
Best validation loss: 2.044344425201416


In [65]:

model.load_state_dict(
    torch.load(
        "checkpoints/sortly_filtered_best.pt",
        map_location=device,
        weights_only=True
    )
)
model.eval()

prompt = (
    "Customer: there is an issue submitting the new delivery address\n"
    "Agent:"
)

print(generate_topk(
    prompt,
    max_new_tokens=80,
    temperature=0.5,
    top_k=10
))


 Customer: there is an issue submitting the new delivery address
Agent: I'm sorry to hear that you're experiencing difficulties while trying to know your delivery address. To assist you further, could you please provide me with the necessary guidance or {{Tracking Number}} or {{Order Number}}? With this information, we will be able to provide you with the most up-to-date information. Your patience is greatly appreciated as we'll be able to us to ensure a smooth experience


In [66]:

@torch.no_grad()
def generate_with_penalty(
    prompt, max_new_tokens=80,
    temperature=0.6, top_k=20, repetition_penalty=1.2
):
    model.eval()
    ids = [BOS_ID] + tokenizer.encode(prompt).ids
    input_ids = torch.tensor([ids], dtype=torch.long, device=device)
    prompt_length = len(ids)

    for _ in range(max_new_tokens):
        context = input_ids[:, -MAX_SEQ_LEN:]

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device == "cuda")
        ):
            logits = model(context)

        scores = logits[0, -1, :].float()
        generated_ids = input_ids[0, prompt_length:].unique()

        for token_id in generated_ids.tolist():
            if scores[token_id] > 0:
                scores[token_id] /= repetition_penalty
            else:
                scores[token_id] *= repetition_penalty

        scores = scores / temperature
        scores[PAD_ID] = -float("inf")
        scores[BOS_ID] = -float("inf")
        scores[UNK_ID] = -float("inf")

        values, indices = torch.topk(scores, min(top_k, scores.numel()))
        probs = F.softmax(values, dim=-1)
        next_id = indices[torch.multinomial(probs, 1)].item()

        if next_id == EOS_ID:
            break

        input_ids = torch.cat([
            input_ids,
            torch.tensor([[next_id]], device=device)
        ], dim=1)

    return tokenizer.decode(input_ids[0].tolist()[1:])

print(generate_with_penalty(
    "Customer: there is an issue submitting the new delivery address\nAgent:"
))



 Customer: there is an issue submitting the new delivery address
Agent: I'm sorry to hear that you're looking for your delivery address. To assist you, please provide me with the details of your new shipping address or any additional information such as the {{Tracking Number}} and give you further? This will help us to assist you in this matter and look forward to locate and ensure a smooth refund.


In [67]:

print("CATEGORY LABELS")
print(train_df["target_category"].value_counts())

print("\nPRIORITY LABELS")
print(train_df["priority"].value_counts())

print("\nSENTIMENT LABELS")
print(train_df["sentiment"].value_counts())

print("\nMissing labels:")
print(
    train_df[
        ["target_category", "priority", "sentiment"]
    ].isna().sum()
)


CATEGORY LABELS
target_category
Account Management    7147
Delivery              3967
Payment               3197
Technical Issue       3197
Refund                2394
Login                  796
Name: count, dtype: int64

PRIORITY LABELS
priority
Medium    19099
High        806
Low         793
Name: count, dtype: int64

SENTIMENT LABELS
sentiment
Neutral     9527
Positive    5878
Negative    5293
Name: count, dtype: int64

Missing labels:
target_category    0
priority           0
sentiment          0
dtype: int64


In [68]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch
import torch.nn as nn

# Use only the customer message as input
X_train_text = train_df["instruction"].astype(str).tolist()
X_val_text = val_df["instruction"].astype(str).tolist()

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_train = vectorizer.fit_transform(X_train_text).toarray()
X_val = vectorizer.transform(X_val_text).toarray()

label_columns = ["target_category", "priority", "sentiment"]
encoders = {}
y_train = {}
y_val = {}

for col in label_columns:
    encoder = LabelEncoder()
    y_train[col] = encoder.fit_transform(train_df[col].astype(str))
    y_val[col] = encoder.transform(val_df[col].astype(str))
    encoders[col] = encoder

print("TF-IDF features:", X_train.shape[1])
for col in label_columns:
    print(f"{col}: {len(encoders[col].classes_)} classes")


TF-IDF features: 5067
target_category: 6 classes
priority: 3 classes
sentiment: 3 classes


In [69]:

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report, f1_score
import numpy as np

class SortlyTicketClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=256):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.category_head = nn.Linear(128, 6)
        self.priority_head = nn.Linear(128, 3)
        self.sentiment_head = nn.Linear(128, 3)

    def forward(self, x):
        features = self.shared(x)
        return (
            self.category_head(features),
            self.priority_head(features),
            self.sentiment_head(features)
        )

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)

train_targets = torch.tensor(
    np.stack([y_train[col] for col in label_columns], axis=1),
    dtype=torch.long
)
val_targets = torch.tensor(
    np.stack([y_val[col] for col in label_columns], axis=1),
    dtype=torch.long
)

train_loader_cls = DataLoader(
    TensorDataset(X_train_tensor, train_targets),
    batch_size=128, shuffle=True
)
val_loader_cls = DataLoader(
    TensorDataset(X_val_tensor, val_targets),
    batch_size=256
)

classifier = SortlyTicketClassifier(
    input_dim=X_train.shape[1]
).to(device)

optimizer_cls = torch.optim.AdamW(
    classifier.parameters(), lr=1e-3, weight_decay=1e-4
)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(1, 11):
    classifier.train()
    total_loss = 0

    for xb, yb in train_loader_cls:
        xb, yb = xb.to(device), yb.to(device)
        optimizer_cls.zero_grad()

        outputs = classifier(xb)
        loss = sum(
            loss_fn(outputs[i], yb[:, i])
            for i in range(3)
        )

        loss.backward()
        optimizer_cls.step()
        total_loss += loss.item()

    print(
        f"Epoch {epoch}/10 | "
        f"Training loss: {total_loss / len(train_loader_cls):.4f}"
    )

# Evaluate on validation data
classifier.eval()
all_preds = []

with torch.no_grad():
    for xb, _ in val_loader_cls:
        outputs = classifier(xb.to(device))
        preds = torch.stack(
            [out.argmax(dim=1).cpu() for out in outputs],
            dim=1
        )
        all_preds.append(preds)

preds = torch.cat(all_preds).numpy()

for i, col in enumerate(label_columns):
    print(f"\n--- {col.upper()} ---")
    print(classification_report(
        y_val[col],
        preds[:, i],
        labels=np.arange(len(encoders[col].classes_)),
        target_names=encoders[col].classes_,
        zero_division=0
    ))
    print(
        "Macro-F1:",
        round(f1_score(y_val[col], preds[:, i], average="macro"), 4)
    )


Epoch 1/10 | Training loss: 1.9569
Epoch 2/10 | Training loss: 0.7567
Epoch 3/10 | Training loss: 0.6904
Epoch 4/10 | Training loss: 0.6528
Epoch 5/10 | Training loss: 0.6266
Epoch 6/10 | Training loss: 0.6070
Epoch 7/10 | Training loss: 0.5904
Epoch 8/10 | Training loss: 0.5721
Epoch 9/10 | Training loss: 0.5580
Epoch 10/10 | Training loss: 0.5440

--- TARGET_CATEGORY ---
                    precision    recall  f1-score   support

Account Management       1.00      1.00      1.00       893
          Delivery       1.00      1.00      1.00       496
             Login       1.00      1.00      1.00       100
           Payment       1.00      0.99      0.99       400
            Refund       1.00      1.00      1.00       299
   Technical Issue       1.00      1.00      1.00       399

          accuracy                           1.00      2587
         macro avg       1.00      1.00      1.00      2587
      weighted avg       1.00      1.00      1.00      2587

Macro-F1: 0.9977

---

In [70]:

import joblib
from pathlib import Path
import json
import torch

artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)

# Save neural network weights and architecture information
torch.save(
    {
        "model_state_dict": classifier.state_dict(),
        "input_dim": X_train.shape[1],
        "hidden_dim": 256,
        "labels": {
            col: encoders[col].classes_.tolist()
            for col in label_columns
        }
    },
    artifact_dir / "sortly_classifier.pt"
)

joblib.dump(vectorizer, artifact_dir / "tfidf_vectorizer.joblib")
joblib.dump(encoders, artifact_dir / "label_encoders.joblib")

print("Saved files:")
for path in artifact_dir.iterdir():
    print(path, f"({path.stat().st_size / 1024:.1f} KB)")


Saved files:
artifacts/sortly_classifier.pt (5207.3 KB)
artifacts/label_encoders.joblib (1.0 KB)
artifacts/tfidf_vectorizer.joblib (201.9 KB)


In [71]:

import torch
import joblib
from pathlib import Path

artifact_dir = Path("artifacts")

# Load saved artifacts
checkpoint = torch.load(
    artifact_dir / "sortly_classifier.pt",
    map_location=device,
    weights_only=True
)
vectorizer = joblib.load(artifact_dir / "tfidf_vectorizer.joblib")
encoders = joblib.load(artifact_dir / "label_encoders.joblib")

# Reconstruct the classifier
classifier = SortlyTicketClassifier(
    input_dim=checkpoint["input_dim"],
    hidden_dim=checkpoint["hidden_dim"]
).to(device)

classifier.load_state_dict(checkpoint["model_state_dict"])
classifier.eval()

def classify_ticket(text):
    features = vectorizer.transform([text]).toarray()
    x = torch.tensor(features, dtype=torch.float32, device=device)

    with torch.no_grad():
        outputs = classifier(x)

    predictions = {}
    for i, col in enumerate(label_columns):
        class_id = outputs[i].argmax(dim=1).item()
        predictions[col] = encoders[col].inverse_transform([class_id])[0]

    return predictions

ticket = "I was charged twice for my order and need a refund urgently."
print("Ticket:", ticket)
print("Predictions:", classify_ticket(ticket))


Ticket: I was charged twice for my order and need a refund urgently.
Predictions: {'target_category': 'Refund', 'priority': 'Medium', 'sentiment': 'Neutral'}


In [72]:

test_ticket = {
    "instruction": "I was charged twice for my order and need a refund urgently.",
    "intent": "payment_issue"
}

print("Current priority:", assign_priority(test_ticket))
print("Current sentiment:", assign_sentiment({
    "instruction": test_ticket["instruction"],
    "response": ""
}))


Current priority: High
Current sentiment: Neutral


In [73]:

import re

def assign_priority(row):
    text = (str(row["instruction"]) + " " + str(row.get("intent", ""))).lower()

    high_terms = [
        "urgent", "urgently", "fraud", "unauthorized",
        "charged twice", "double charged", "payment_issue",
        "cannot access", "locked out", "security breach",
        "stolen", "account compromised"
    ]
    low_terms = [
        "newsletter", "review", "feedback", "suggestion",
        "feature request"
    ]

    if any(term in text for term in high_terms):
        return "High"
    if any(term in text for term in low_terms):
        return "Low"
    return "Medium"


def assign_sentiment(row):
    text = (
        str(row.get("instruction", "")) + " " +
        str(row.get("response", ""))
    ).lower()

    negative_terms = [
        "angry", "frustrated", "disappointed", "unhappy",
        "terrible", "awful", "hate", "complaint",
        "charged twice", "wrong charge", "not working",
        "failed", "cannot", "can't", "unable", "error",
        "problem", "issue", "broken", "unacceptable",
        "refund", "cancel", "lost", "delayed"
    ]
    positive_terms = [
        "thank you", "thanks", "great", "excellent",
        "happy", "love", "wonderful", "satisfied",
        "appreciate"
    ]

    if any(term in text for term in negative_terms):
        return "Negative"
    if any(term in text for term in positive_terms):
        return "Positive"
    return "Neutral"


test_row = {
    "instruction": "I was charged twice for my order and need a refund urgently.",
    "intent": "payment_issue",
    "response": ""
}

print("Updated priority:", assign_priority(test_row))
print("Updated sentiment:", assign_sentiment(test_row))


Updated priority: High
Updated sentiment: Negative


In [74]:

# Recalculate priority and sentiment consistently across all splits
for split_df in [train_df, val_df, test_df]:
    split_df["priority"] = split_df.apply(assign_priority, axis=1)

    split_df["sentiment"] = split_df.apply(
        lambda row: assign_sentiment({
            "instruction": row["instruction"],
            "response": ""
        }),
        axis=1
    )

print("Updated training priority labels:")
print(train_df["priority"].value_counts())

print("\nUpdated training sentiment labels:")
print(train_df["sentiment"].value_counts())

print("\nExample ticket labels:")
print("Priority:", assign_priority(test_row))
print("Sentiment:", assign_sentiment(test_row))


Updated training priority labels:
priority
Medium    19098
High        807
Low         793
Name: count, dtype: int64

Updated training sentiment labels:
sentiment
Neutral     16040
Negative     4637
Positive       21
Name: count, dtype: int64

Example ticket labels:
Priority: High
Sentiment: Negative


In [75]:

def assign_sentiment(row):
    text = str(row.get("instruction", "")).lower()

    negative_patterns = [
        r"\bangry\b", r"\bfrustrated\b", r"\bdisappointed\b",
        r"\bunhappy\b", r"\bterrible\b", r"\bawful\b",
        r"\bhate\b", r"\bscam\b", r"\bunacceptable\b",
        r"\bcharged twice\b", r"\bwrong charge\b",
        r"\bvery poor\b", r"\bnot satisfied\b",
        r"\bnever again\b", r"\bcomplaint\b"
    ]

    positive_patterns = [
        r"\bthank you\b", r"\bthanks\b", r"\bgreat\b",
        r"\bexcellent\b", r"\bhappy\b", r"\blove\b",
        r"\bwonderful\b", r"\bsatisfied\b",
        r"\bappreciate\b", r"\bwell done\b"
    ]

    if any(re.search(p, text) for p in negative_patterns):
        return "Negative"
    if any(re.search(p, text) for p in positive_patterns):
        return "Positive"
    return "Neutral"


for split_df in [train_df, val_df, test_df]:
    split_df["sentiment"] = split_df.apply(
        lambda row: assign_sentiment({
            "instruction": row["instruction"]
        }),
        axis=1
    )

print(train_df["sentiment"].value_counts())



sentiment
Neutral     20422
Negative      263
Positive       13
Name: count, dtype: int64


In [76]:

print("CATEGORY DISTRIBUTION")
print(train_df["target_category"].value_counts())

print("\nPRIORITY DISTRIBUTION")
print(train_df["priority"].value_counts())

print("\nSENTIMENT LABELS")
print(train_df["sentiment"].value_counts())


CATEGORY DISTRIBUTION
target_category
Account Management    7147
Delivery              3967
Payment               3197
Technical Issue       3197
Refund                2394
Login                  796
Name: count, dtype: int64

PRIORITY DISTRIBUTION
priority
Medium    19098
High        807
Low         793
Name: count, dtype: int64

SENTIMENT LABELS
sentiment
Neutral     20422
Negative      263
Positive       13
Name: count, dtype: int64


In [77]:

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score
import numpy as np

# Encode only the two labels we are keeping
category_encoder = LabelEncoder()
priority_encoder = LabelEncoder()

y_cat_train = category_encoder.fit_transform(train_df["target_category"])
y_cat_val = category_encoder.transform(val_df["target_category"])

y_pri_train = priority_encoder.fit_transform(train_df["priority"])
y_pri_val = priority_encoder.transform(val_df["priority"])

targets_train = torch.tensor(
    np.stack([y_cat_train, y_pri_train], axis=1),
    dtype=torch.long
)
targets_val = torch.tensor(
    np.stack([y_cat_val, y_pri_val], axis=1),
    dtype=torch.long
)

train_loader_2 = DataLoader(
    TensorDataset(X_train_tensor, targets_train),
    batch_size=128, shuffle=True
)
val_loader_2 = DataLoader(
    TensorDataset(X_val_tensor, targets_val),
    batch_size=256
)

class SortlyClassifierV2(nn.Module):
    def __init__(self, input_dim, n_categories, n_priorities):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.category_head = nn.Linear(128, n_categories)
        self.priority_head = nn.Linear(128, n_priorities)

    def forward(self, x):
        h = self.shared(x)
        return self.category_head(h), self.priority_head(h)

classifier_v2 = SortlyClassifierV2(
    X_train.shape[1],
    len(category_encoder.classes_),
    len(priority_encoder.classes_)
).to(device)

optimizer_v2 = torch.optim.AdamW(
    classifier_v2.parameters(), lr=1e-3, weight_decay=1e-4
)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(1, 11):
    classifier_v2.train()
    total_loss = 0

    for xb, yb in train_loader_2:
        xb, yb = xb.to(device), yb.to(device)
        optimizer_v2.zero_grad()

        cat_logits, pri_logits = classifier_v2(xb)
        loss = loss_fn(cat_logits, yb[:, 0]) + loss_fn(pri_logits, yb[:, 1])

        loss.backward()
        optimizer_v2.step()
        total_loss += loss.item()

    print(f"Epoch {epoch}/10 | Loss: {total_loss / len(train_loader_2):.4f}")

# Validation evaluation
classifier_v2.eval()
preds_cat, preds_pri = [], []

with torch.no_grad():
    for xb, _ in val_loader_2:
        cat_logits, pri_logits = classifier_v2(xb.to(device))
        preds_cat.extend(cat_logits.argmax(1).cpu().tolist())
        preds_pri.extend(pri_logits.argmax(1).cpu().tolist())

print("\nCATEGORY")
print(classification_report(
    y_cat_val, preds_cat,
    labels=np.arange(len(category_encoder.classes_)),
    target_names=category_encoder.classes_,
    zero_division=0
))
print("Category macro-F1:", f1_score(y_cat_val, preds_cat, average="macro"))

print("\nPRIORITY")
print(classification_report(
    y_pri_val, preds_pri,
    labels=np.arange(len(priority_encoder.classes_)),
    target_names=priority_encoder.classes_,
    zero_division=0
))
print("Priority macro-F1:", f1_score(y_pri_val, preds_pri, average="macro"))


Epoch 1/10 | Loss: 1.0162
Epoch 2/10 | Loss: 0.0199
Epoch 3/10 | Loss: 0.0071
Epoch 4/10 | Loss: 0.0036
Epoch 5/10 | Loss: 0.0037
Epoch 6/10 | Loss: 0.0022
Epoch 7/10 | Loss: 0.0015
Epoch 8/10 | Loss: 0.0012
Epoch 9/10 | Loss: 0.0014
Epoch 10/10 | Loss: 0.0012

CATEGORY
                    precision    recall  f1-score   support

Account Management       1.00      1.00      1.00       893
          Delivery       1.00      1.00      1.00       496
             Login       1.00      1.00      1.00       100
           Payment       1.00      1.00      1.00       400
            Refund       1.00      1.00      1.00       299
   Technical Issue       1.00      1.00      1.00       399

          accuracy                           1.00      2587
         macro avg       1.00      1.00      1.00      2587
      weighted avg       1.00      1.00      1.00      2587

Category macro-F1: 0.9985717058955398

PRIORITY
              precision    recall  f1-score   support

        High       0.99

In [78]:

import joblib
import torch
from pathlib import Path

artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)

torch.save(
    {
        "model_state_dict": classifier_v2.state_dict(),
        "input_dim": X_train.shape[1],
        "n_categories": len(category_encoder.classes_),
        "n_priorities": len(priority_encoder.classes_),
        "category_labels": category_encoder.classes_.tolist(),
        "priority_labels": priority_encoder.classes_.tolist()
    },
    artifact_dir / "sortly_classifier_v2.pt"
)

joblib.dump(
    vectorizer,
    artifact_dir / "sortly_tfidf_vectorizer.joblib"
)
joblib.dump(
    category_encoder,
    artifact_dir / "sortly_category_encoder.joblib"
)
joblib.dump(
    priority_encoder,
    artifact_dir / "sortly_priority_encoder.joblib"
)

print("Saved artifacts:")
for path in sorted(artifact_dir.iterdir()):
    print(f"{path.name}: {path.stat().st_size / 1024:.1f} KB")


Saved artifacts:
label_encoders.joblib: 1.0 KB
sortly_category_encoder.joblib: 0.5 KB
sortly_classifier.pt: 5207.3 KB
sortly_classifier_v2.pt: 5205.2 KB
sortly_priority_encoder.joblib: 0.5 KB
sortly_tfidf_vectorizer.joblib: 201.9 KB
tfidf_vectorizer.joblib: 201.9 KB


In [79]:

import torch
import joblib
from pathlib import Path

artifact_dir = Path("artifacts")

# Load the saved artifacts
checkpoint = torch.load(
    artifact_dir / "sortly_classifier_v2.pt",
    map_location=device,
    weights_only=True
)
vectorizer = joblib.load(
    artifact_dir / "sortly_tfidf_vectorizer.joblib"
)
category_encoder = joblib.load(
    artifact_dir / "sortly_category_encoder.joblib"
)
priority_encoder = joblib.load(
    artifact_dir / "sortly_priority_encoder.joblib"
)

# Rebuild the saved model
classifier_v2 = SortlyClassifierV2(
    input_dim=checkpoint["input_dim"],
    n_categories=checkpoint["n_categories"],
    n_priorities=checkpoint["n_priorities"]
).to(device)

classifier_v2.load_state_dict(checkpoint["model_state_dict"])
classifier_v2.eval()

def predict_ticket(text):
    features = vectorizer.transform([text]).toarray()
    x = torch.tensor(features, dtype=torch.float32, device=device)

    with torch.no_grad():
        category_logits, priority_logits = classifier_v2(x)

    category = category_encoder.inverse_transform(
        [category_logits.argmax(1).item()]
    )[0]
    priority = priority_encoder.inverse_transform(
        [priority_logits.argmax(1).item()]
    )[0]

    return {"category": category, "priority": priority}

tickets = [
    "I cannot sign in after resetting my password.",
    "My parcel has not arrived and tracking has not updated.",
    "I see two charges for one purchase and need urgent help.",
    "Please explain how I can download my invoice.",
    "I want to delete my account permanently."
]

for ticket in tickets:
    print("\nTicket:", ticket)
    print("Prediction:", predict_ticket(ticket))



Ticket: I cannot sign in after resetting my password.
Prediction: {'category': 'Login', 'priority': 'Medium'}

Ticket: My parcel has not arrived and tracking has not updated.
Prediction: {'category': 'Delivery', 'priority': 'Medium'}

Ticket: I see two charges for one purchase and need urgent help.
Prediction: {'category': 'Account Management', 'priority': 'Medium'}

Ticket: Please explain how I can download my invoice.
Prediction: {'category': 'Payment', 'priority': 'Medium'}

Ticket: I want to delete my account permanently.
Prediction: {'category': 'Account Management', 'priority': 'Medium'}


In [80]:

import re

def predict_ticket(text):
    features = vectorizer.transform([text]).toarray()
    x = torch.tensor(features, dtype=torch.float32, device=device)

    with torch.no_grad():
        category_logits, priority_logits = classifier_v2(x)

    category = category_encoder.inverse_transform(
        [category_logits.argmax(1).item()]
    )[0]
    priority = priority_encoder.inverse_transform(
        [priority_logits.argmax(1).item()]
    )[0]

    normalized = text.lower()

    # Business rules for clear, high-risk payment cases
    duplicate_charge = any(term in normalized for term in [
        "charged twice", "double charged", "duplicate charge",
        "two charges", "charged two times"
    ])

    urgent_terms = any(term in normalized for term in [
        "urgent", "urgently", "fraud", "unauthorized",
        "account compromised", "stolen card"
    ])

    if duplicate_charge:
        category = "Payment"

    if duplicate_charge or urgent_terms:
        priority = "High"

    return {"category": category, "priority": priority}


test_ticket = "I see two charges for one purchase and need urgent help."
print("Ticket:", test_ticket)
print("Prediction:", predict_ticket(test_ticket))



Ticket: I see two charges for one purchase and need urgent help.
Prediction: {'category': 'Payment', 'priority': 'High'}
